In [ ]:
import os
import ollama
import kagglehub
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from IPython.display import Markdown, display

In [ ]:
# Load environment variables
load_dotenv()
api_key = os.getenv("OPEN_API_KEY")

In [ ]:
!pip install kagglehub, pypdf

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("manisha717/dataset-of-pdf-files")

print("Path to dataset files:", path)

In [ ]:
class ResumePdf:

    def __init__(self, system_prompt: str = None, model_by_api: str = 'gpt'):
        if system_prompt is None:
            self.system_prompt = "Eres un escritor experto, sabes a la perfección cómo resumir cualquier tipo de texto.\
                    Tu misión es resumir con claridad, de manera que resaltes los puntos claves, el autor, la fecha y sobre \
                    todo, debes ser capaz de conservar el sentido y sentimiento del texto según el área de conocimiento \
                    (podría ser una carta de renuncia, una carta de amor, una carta para informar sobre avances \
                    en un negocio, un paper científico, etc.) \
                    Es altamente probable que la información esté en inglés, por tanto, debes traducir al español siempre.\
                    Responde en formato markdown."
        else:
            self.system_prompt = system_prompt

        self.model_by_api = model_by_api
        self.ollama_api = "http://localhost:11434/api/chat"

    def get_text_pdf(self, path):
        reader = PdfReader(path)
        full_text = '\n'.join((page.extract_text() for i, page in enumerate(reader.pages) if i != 0))
        return full_text

    def pdf_context(self, text: str) -> str:
        user_prompt = "Por favor, resumen el contenido del \
            pdf en máximo 500 palabras. Trata de obtener la idea global y describir las secciones \
            más importantes del texto\n\n"
        user_prompt += f"{text}"

        return user_prompt

    def messages_for(self, path_pdf):
        full_text = self.get_text_pdf(path=path_pdf)
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": self.pdf_context(text=full_text)}
        ]
    
    def display_summary(self, summary):
        display(Markdown(summary))
    
    def summarize_pdf(self, path_pdf):
        messages = self.messages_for(path_pdf)

        if self.model_by_api == 'gpt':
            client = OpenAI(api_key=api_key)
            response = client.chat.completions.create(model="gpt-4o-mini", 
                                                      messages=messages)
            
            summary = response.choices[0].message.content

        if self.model_by_api == 'ollama_local':
            response = ollama.chat(model="llama3.2", 
                                   messages=messages)
            summary = response['message']['content']
        
        if self.model_by_api == 'ollama_openai':
            ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', 
                                       api_key='ollama')
            response = ollama_via_openai.chat.completions.create(
                            model="llama3.2",
                            messages=messages
                        )
            summary = response.choices[0].message.content
            
        return summary
    
    def generate_resumens(self, lista_paths_pdf):
        for path_pdf in lista_paths_pdf:
            try:
                summary = self.summarize_pdf(path_pdf=path_pdf)
                self.display_summary(summary) 
                yield {
                    "path": path_pdf,
                    "summary": summary
                }
            except Exception as e:
                self.display_summary(f'## Error {e} with document {path_pdf}')
                yield {
                    "path": path_pdf,
                    "error": str(e)
                }

In [ ]:
path_pdfs_database = "path/to/pdf_bank"
all_entries = os.listdir(path_pdfs_database)
files = [entry for entry in all_entries if os.path.isfile(os.path.join(path_pdfs_database, entry))]
files_path = [os.path.join(path_pdfs_database, f) for f in files]

model_api = 'gpt'  # o "ollama_local", "ollama_openai"
resumidor = ResumePdf(model_by_api=model_api)

resumens = resumidor.generate_resumens(files_path)

In [ ]:
sum = next(resumens)

In [ ]:
sum

In [ ]:
sum = next(resumens)

In [ ]:
sum

In [ ]:
sum = next(resumens)
sum

In [ ]:
# Store summaries for postprocessing

path_pdfs_database = "C:/Users/mario/.cache/kagglehub/datasets/manisha717/dataset-of-pdf-files/versions/1/Pdf/"
all_entries = os.listdir(path_pdfs_database)
files = [entry for entry in all_entries if os.path.isfile(os.path.join(path_pdfs_database, entry))]
files_path = [os.path.join(path_pdfs_database, f) for f in files]

model_api = 'gpt'  # o "ollama_local", "ollama_openai"
resumidor = ResumePdf(model_by_api=model_api)

resumens = resumidor.generate_resumens(files_path)

summaries = {}

for i, resumen in enumerate(resumens):
    if 'error' in resumen:
        print(f"❌ Error procesando {resumen['path']}: {resumen['error']}")
    else:
        print(f"✅ {resumen['path']}:\n")
        summaries[resumen['path']] = resumen['summary']
        print("\n" + "="*80 + "\n")

    if i == 3:
        break